In [ ]:
import sys
import os
import gc
import zipfile
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import numpy.lib.format as npformat  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402
from mne.time_frequency import tfr_array_morlet  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
    SingleDataMetadata,
    ConditionVariants,
    MusicTypeVariants,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.preprocessing.marker_shift import load_marker_shift_table  # noqa: E402
from src.preprocessing.stimulus_alignment import (  # noqa: E402
    DEFAULT_STIMULUS_LABEL,
    get_stimulus_onset_samples,
    resolve_stimulus_marker,
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# ASSR Stimulus Timing — Is the Onset Where It Should Be?

The `fam+` stimulus markers of the ASSR recordings were read **late** out of the
EDFs, by a constant that differs from recording to recording (372-455 ms; the
cause and the exact per-recording values are in
[`assr_annotation_discrepancy.ipynb`](assr_annotation_discrepancy.ipynb)). The
correction is now applied per recording, and the stimulus-aligned products —
`concatenated/Placebo_ASSR.npy` with its `.stimulus_onsets.npy`, and the wavelet
cache built from it — have been regenerated with it.

This notebook is the **acceptance test** for that regeneration. It asks the
question from the other side: *forget the annotations, look at the data — does
the ASSR response sit where the stored onsets say it does?*

**Two failures are possible and they need different instruments.**

| failure | what it looks like | instrument | resolution |
|---|---|---|---|
| **global** — every recording shifted by the same amount | the driven response sits off-centre from the nominal `[0, 500] ms` window in *all* of them | 40 Hz **inter-trial coherence envelope**: where the response starts and stops | ~±50 ms |
| **relative / within pairs** — recordings shifted by *different* amounts | the group envelope can still look fine; individual recordings disagree with each other | 40 Hz **evoked phase clustering across recordings** | ~±3 ms |

"Shifted relative to each other" is asked of every pairing that could hide a
problem: any **two recordings** of a group (step 3), the **two halves of one
recording's stimulus sequence** (step 3), and the **two sessions of one
participant**, placebo against psilocybin (step 6).

The second instrument is what makes the per-recording question answerable. The
ASSR is phase-locked to the stimulus, so at 40 Hz a timing error of Δ ms rotates
a recording's response phase by `Δ / 25 ms` of a cycle. If every recording is
timed correctly, their phases pile up on one direction (only the physiological
transmission delay, which is common to all, remains); if they are timed
individually wrong, the phases scatter. A 22 ms spread — exactly what the old
single global offset would have left behind — is nearly a full cycle and
destroys the clustering completely.

**Steps.**
1. Global — where does the driven response sit relative to the stored onsets?
2. Per recording, coarse — the blind look: is any recording visibly displaced?
3. Per recording, fine — 40 Hz phase clustering across recordings, and every
   pairwise relative lag.
4. Positive controls — re-inject the pre-fix error and the estimator's
   resolution curve, to show these tests would have caught it.
5. The wavelet cache — does it carry the same time base as the array it was
   built from?
6. Within-participant **session pairs** (Placebo vs Psilocybin, on continuous
   `RAW_AFTER_ICA`) — a timing difference between the two sessions of one
   participant would masquerade as a drug effect.
7. Verdict table.

**What "pass" means.** The response is *expected* to start slightly after the
acoustic onset: the auditory transmission delay puts the driven 40 Hz activity
some 30-50 ms late, and that delay is physiology, not a timing error. A marker
error is distinguished from it by two things — its size (hundreds of
milliseconds), and the fact that it moves the response's *leading and trailing
edge together* while leaving its width unchanged. So the pass criterion is a
fitted response window of the right **width** (the 500 ms train), placed within a
few tens of milliseconds after zero, with **no recording-to-recording spread**.

## Configuration

In [ ]:
# ── Group under test ─────────────────────────────────────────────────────────
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO  # project default; the concatenated group
MUSIC_TYPE = MusicTypeVariants.ASSR
STIMULUS_LABEL = DEFAULT_STIMULUS_LABEL  # "fam+"

# ── Read-out ─────────────────────────────────────────────────────────────────
ASSR_FREQ = 40.0  # the driving frequency; the whole read-out lives here
N_CYCLES = 4.0  # 4 cycles @ 40 Hz = a 100 ms kernel -> ~±50 ms of smearing.
# The kernel is symmetric, so it blurs the response edges but cannot displace
# them: a lag measured through it is unbiased.

# Stimulus-locked window (s, relative to each stored onset). Asymmetric on
# purpose: the response is expected in [0, 0.5] s, and the window must extend
# far enough past it to see the response *stop*, which is half the evidence
# that the box being fitted is the stimulus train and not a noise excursion.
EPOCH_PRE_S = 0.5
EPOCH_POST_S = 0.9

# Sub-windows read off the profile. The plateau excludes the first/last ~120 ms
# of the train because the Morlet kernel smears the edges into them; the
# baseline stops 150 ms before zero for the same reason.
PLATEAU_S = (0.12, 0.45)
BASELINE_S = (-EPOCH_PRE_S, -0.15)

# ── Search grids for the envelope fit ────────────────────────────────────────
# SYMMETRIC about zero by design: a grid skewed to one side would build the
# expected answer into the estimate.
LAG_GRID_S = np.arange(-0.2, 0.2005, 0.004)
DUR_GRID_S = np.arange(0.30, 0.705, 0.01)

# ── Pass/fail tolerances ─────────────────────────────────────────────────────
# Global: the fitted onset must land within this of zero. 100 ms comfortably
# covers the physiological transmission delay (~30-50 ms) and the fit's own
# resolution, while being far below the 372-455 ms error under test.
TOL_GLOBAL_LAG_S = 0.100
# ... and the fitted width must match the paradigm's train to within this.
TOL_DURATION_S = 0.100
# Per recording, the phase read-out resolves timing to a millisecond or two —
# fine enough that it also picks up genuine *physiological* differences in
# auditory transmission latency between people, which are of the same size and
# which no measurement on these data can separate from a timing error. The
# criteria are therefore chosen not to conflate the two:
#
# (i) the SPREAD across recordings. A per-recording timing error would have to
#     inflate it; the pre-fix data would show ~22 ms SD, individual latency
#     variation contributes only a few ms.
TOL_RELATIVE_SPREAD_S = 0.010
# (ii) whether the deviations still TRACK the calibration residual. Physiology
#     cannot know what a recording's EDF header rounded away, so a non-zero
#     slope here is timing and nothing else. 1.0 = entirely uncorrected.
TOL_RESIDUAL_SLOPE = 0.3
# (iii) a per-recording flag, informational only for the reason above: a single
#     recording sitting further out than this is worth a look, not a failure.
WATCH_RELATIVE_LAG_S = 0.005
# Within one recording, and between the two sessions of one participant, the
# physiological confound is much weaker (same ears, same head), so these are
# hard criteria.
TOL_DRIFT_S = 0.005
TOL_PAIR_BIAS_S = 0.005
# Coarse per-recording lags are noisy (see step 2); this only catches gross
# displacement, not the millisecond question.
TOL_ENVELOPE_LAG_S = 0.150

# ── Optional / heavier steps ─────────────────────────────────────────────────
# Step 4b: per-recording jitter (ms, SD) injected to trace the phase test's
# resolution. Each level costs one epoching pass over the array (~15 s).
JITTER_SWEEP_MS = [0, 2, 4, 8, 16]
JITTER_REPEATS = 3  # random draws averaged per level: with 15 recordings a single
# draw of the jitter is itself noisy enough to make the curve wobble
# Step 5 streams the whole wavelet cache once (~5 min for a 52 GB npz): the
# file is a single deflate stream and cannot be sliced randomly.
RUN_WAVELET_CHECK = True
# Step 6 re-reads every RAW_AFTER_ICA recording (~3 min for 38 files).
RUN_SESSION_PAIR_CHECK = True
PAIR_CHANNEL_STRIDE = 2  # channel subsample for step 6; timing is not a
# topography question, so a spatial subsample buys speed at no cost

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "assr_stimulus_timing_verification"
    / f"{CONDITION.value}_{MUSIC_TYPE.value}"
)
if SAVE_PLOTS:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Group        : {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Read-out     : {ASSR_FREQ:.0f} Hz, {N_CYCLES:.0f}-cycle Morlet")
print(f"Epoch window : [{-EPOCH_PRE_S:+.2f}, {EPOCH_POST_S:+.2f}] s around each stored onset")
print(f"Paradigm     : {AssrEpoch.STIMULUS_DURATION_S * 1e3:.0f} ms train")
print(f"Plots        -> {PLOTS_DIR if SAVE_PLOTS else '(not saved)'}")

## Helper Functions

Two read-outs, and they answer different questions.

**`itc_profile` — the envelope.** Inter-trial phase coherence at 40 Hz measures
*how reproducibly* the response follows the stimulus from trial to trial. It
rises when the train starts driving the cortex and falls when it stops, so its
profile brackets the stimulus in time. It is an **absolute** read-out — it says
where the response is — but a coarse one, because its edges are smeared by the
kernel and buried in single-recording noise.

**`evoked_phasors` — the phase.** The complex 40 Hz coefficient of the
trial-averaged response. Its *angle* is the phase of the steady-state response
relative to the marker, which shifts by a full cycle for every 25 ms of timing
error. That makes it exquisitely sensitive but **relative** and **wrapped**: it
compares recordings to each other, and it cannot tell 0 ms from 25 ms. Used
together the two cover each other's blind spot — the envelope excludes gross
displacement, the phase resolves what is left to a few milliseconds.

`fit_box` is the estimator applied to an envelope: a matched filter for a
rectangular response, scanned over a lag grid symmetric about zero. It uses the
whole profile rather than a single peak sample, so it degrades gracefully as the
SNR drops, and the width it returns is the sanity check — a fit that lands on
noise does not come back 500 ms wide.

In [ ]:
def stimulus_epochs(data, onsets, pre, post):
    """Windows around every stored onset that fits inside the recording.

    :param data: ``(n_channels, n_times)`` array; the time axis is last.
    :param onsets: Onset sample indices *into this array*.
    :param pre: Samples kept before each onset.
    :param post: Samples kept after each onset.
    :return: ``(n_epochs, n_channels, pre + post)`` array.
    """
    n_times = data.shape[-1]
    keep = [int(o) for o in onsets if o - pre >= 0 and o + post <= n_times]
    if not keep:
        raise ValueError("No onset window fits inside the recording.")
    return np.stack([np.asarray(data[:, o - pre : o + post], dtype=float) for o in keep])


def morlet_at(signals, sfreq, freq=None, n_cycles=None):
    """Complex Morlet coefficient at one frequency.

    :param signals: ``(n_epochs, n_channels, n_times)`` array.
    :return: ``(n_epochs, n_channels, n_times)`` complex array.
    """
    tfr = tfr_array_morlet(
        signals,
        sfreq,
        [freq or ASSR_FREQ],
        n_cycles=n_cycles or N_CYCLES,
        output="complex",
        verbose=False,
    )
    # Collapse any singleton frequency / taper axes MNE may add.
    return tfr.reshape(tfr.shape[0], tfr.shape[1], -1, tfr.shape[-1])[:, :, 0, :]


def itc_from_coefficients(z):
    """Inter-trial phase coherence from per-trial complex coefficients.

    :param z: ``(n_epochs, n_channels, n_times)`` complex coefficients.
    :return: ``(n_channels, n_times)`` coherence in [0, 1].
    """
    unit = z / (np.abs(z) + 1e-30)  # unit modulus: phase only, amplitude discarded
    return np.abs(unit.mean(axis=0))


def evoked_phasors(epochs, sfreq, freq=None, n_cycles=None):
    """Complex ``freq`` coefficient of the trial-averaged (evoked) response.

    The Morlet transform is linear, so averaging the epochs first and
    transforming once is *identical* to transforming every epoch and averaging —
    and cheaper by a factor of the trial count.

    :param epochs: ``(n_epochs, n_channels, n_times)`` array.
    :return: ``(n_channels, n_times)`` complex array.
    """
    return morlet_at(epochs.mean(axis=0)[None], sfreq, freq, n_cycles)[0]


def fit_box(times, profile, lag_grid=None, dur_grid=None):
    """Best-fitting boxcar — a matched filter for a rectangular response.

    :param times: ``(n_times,)`` time axis in seconds, 0 at the stored onset.
    :param profile: ``(n_times,)`` envelope (e.g. an ITC time course).
    :param lag_grid: Candidate onset lags in seconds. Keep it symmetric about 0.
    :param dur_grid: Candidate durations in seconds; a scalar fixes the width.
    :return: ``(lag, duration, score)`` of the best fit.
    """
    lag_grid = LAG_GRID_S if lag_grid is None else np.atleast_1d(lag_grid)
    dur_grid = DUR_GRID_S if dur_grid is None else np.atleast_1d(dur_grid)
    centred = profile - profile.mean()
    best = (-np.inf, np.nan, np.nan)
    for duration in dur_grid:
        for lag in lag_grid:
            box = ((times >= lag) & (times < lag + duration)).astype(float)
            box -= box.mean()
            norm = np.linalg.norm(box)
            if norm == 0:
                continue  # box covers the whole window: no contrast to fit
            score = float(box @ centred / norm)
            if score > best[0]:
                best = (score, float(lag), float(duration))
    return best[1], best[2], best[0]


def half_max_edges(times, profile, baseline_mask, plateau_mask):
    """Rise / fall times where a profile crosses half its baseline->plateau step.

    An estimator independent of :func:`fit_box`, assuming nothing about the
    response being rectangular. Crossings are taken from the plateau outwards,
    so a spurious Morlet spike at an epoch edge cannot invert the window.

    :return: ``(rise, fall)`` in seconds; ``nan`` where no crossing exists.
    """
    level = 0.5 * (profile[baseline_mask].mean() + profile[plateau_mask].mean())
    centre = int(np.flatnonzero(plateau_mask).mean())
    rise = fall = np.nan
    below_left = np.flatnonzero(profile[:centre] < level)
    if len(below_left):
        i = int(below_left[-1])
        rise = float(np.interp(level, profile[i : i + 2], times[i : i + 2]))
    below_right = np.flatnonzero(profile[centre:] < level)
    if len(below_right):
        j = centre + int(below_right[0])
        rise_pair = [profile[j], profile[j - 1]]
        fall = float(np.interp(level, rise_pair, [times[j], times[j - 1]]))
    return rise, fall


def phase_lag_s(phasors, template, freq=None):
    """Timing difference between a recording's response phase and a template.

    Every (channel, time) cell votes, weighted by how strong the response is in
    both — a channel with no ASSR carries no timing information and must not be
    allowed to contribute noise on equal terms.

    :param phasors: ``(n_channels, n_times)`` complex coefficients of one recording.
    :param template: ``(n_channels, n_times)`` complex reference.
    :return: Lag in seconds, wrapped into ``±1 / (2 * freq)`` (±12.5 ms at 40 Hz).
    """
    freq = freq or ASSR_FREQ
    difference = phasors * np.conj(template)
    weights = np.abs(phasors) * np.abs(template)
    unit = difference / (np.abs(difference) + 1e-30)
    return float(np.angle((weights * unit).sum()) / (2 * np.pi * freq))


def phase_locking_value(phasors, axis=0):
    """Resultant length of the phases along ``axis`` (1 = identical, 0 = scattered).

    Chance level for ``n`` independent phases is ``1 / sqrt(n)``, not 0.
    """
    unit = phasors / (np.abs(phasors) + 1e-30)
    return np.abs(unit.mean(axis=axis))


def circular_spread_s(resultant, freq=None, n=None):
    """The timing spread implied by a resultant length, in seconds.

    :param resultant: Phase-locking value across ``n`` recordings.
    :param n: Number of recordings; supplied, the chance-level resultant is
        removed first so the spread is not underestimated at low ``n``.
    :return: Circular SD converted to seconds at ``freq``.
    """
    freq = freq or ASSR_FREQ
    resultant = float(np.clip(resultant, 1e-12, 1.0))
    if n:  # de-bias: n independent phases already give 1/sqrt(n) by chance
        resultant = float(np.clip((resultant**2 * n - 1) / (n - 1), 1e-12, 1.0)) ** 0.5
    return float(np.sqrt(-2.0 * np.log(resultant)) / (2 * np.pi * freq))


def wrap_to_cycle(seconds, freq=None):
    """Fold a duration into ``±half a cycle`` of ``freq``.

    The phase read-out cannot see more than one cycle, so a *predicted* timing
    error has to be folded the same way before it can be compared against a
    measured one. Without this, a prediction spanning several cycles is compared
    against a measurement that never can.

    :param seconds: Duration(s) in seconds.
    :return: The equivalent duration(s) in ``[-1/(2*freq), +1/(2*freq))``.
    """
    period = 1.0 / (freq or ASSR_FREQ)
    return (np.asarray(seconds) + period / 2) % period - period / 2


def ms(seconds):
    """Format a duration in seconds as a signed millisecond string."""
    return "nan" if not np.isfinite(seconds) else f"{seconds * 1e3:+.0f} ms"


print("Helpers defined.")

## Data Loading

Everything the acceptance test needs comes from the products under test
themselves — the concatenated array and the onset positions saved beside it —
plus two pieces of provenance: the concatenation metadata (which recording is
which row) and the calibration table that produced the correction. The
calibration is **not** used to time anything here; it is the ground truth the
blind measurements are compared against in step 4.

In [ ]:
safe_label = f"{CONDITION.value}_{MUSIC_TYPE.value}"
concat_dir = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / PreprocessedDataVariants.CONCATENATED.value
)
array_path = concat_dir / f"{safe_label}.npy"
onsets_path = concat_dir / f"{safe_label}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
metadata_path = concat_dir / f"{safe_label}.metadata.csv"

wavelet_dir = ProjectPaths.PROCESSED_DATA_DIR / EXPERIMENT.value / "wavelets" / "broadband"
wavelet_paths = sorted(wavelet_dir.glob(f"{safe_label}__wavelet_power__*__freqdim1.npz"))
for path in (array_path, onsets_path, metadata_path):
    assert path.exists(), f"Missing expected file: {path}"

# Memory-mapped: only the subject currently being measured is ever in RAM.
concatenated = np.load(array_path, mmap_mode="r")  # (n_subjects, n_channels, n_times)
onsets = np.load(onsets_path)  # shared onset sample indices
metadata = pd.read_csv(metadata_path, index_col=0)
n_subjects, n_channels, n_times = concatenated.shape

# The sampling rate is read from the wavelet cache rather than assumed: the two
# products must share a time base, and this is the one place it is recorded.
if wavelet_paths:
    with zipfile.ZipFile(wavelet_paths[-1]) as archive:
        with archive.open("sfreq.npy") as handle:
            sfreq = float(npformat.read_array(handle, allow_pickle=True))
        with archive.open("freqs.npy") as handle:
            wavelet_freqs = npformat.read_array(handle, allow_pickle=True)
        with archive.open("feature_names.npy") as handle:
            feature_names = npformat.read_array(handle, allow_pickle=True)
    channel_names = [
        str(name).split("@")[0] for name in feature_names[:: len(wavelet_freqs)]
    ]
else:
    sfreq, wavelet_freqs, channel_names = 250.0, None, [f"#{i}" for i in range(n_channels)]
    print("WARNING: no wavelet cache found; sampling rate assumed to be 250 Hz.")

PRE = int(round(EPOCH_PRE_S * sfreq))
POST = int(round(EPOCH_POST_S * sfreq))
epoch_times = np.arange(-PRE, POST) / sfreq
plateau_mask = (epoch_times >= PLATEAU_S[0]) & (epoch_times < PLATEAU_S[1])
baseline_mask = (epoch_times >= BASELINE_S[0]) & (epoch_times < BASELINE_S[1])

# Row -> recording, and the calibrated offset that timed that recording.
index_column = str(SingleDataMetadata.CONCATENATED_PERSON_INDEX)
filename_column = str(SingleDataMetadata.FILENAME)
participant_column = str(SingleDataMetadata.PARTICIPANT_ID)
stem_by_row = {
    int(row[index_column]): Path(row[filename_column]).stem
    for _, row in metadata.iterrows()
}
participant_by_row = {
    int(row[index_column]): str(row[participant_column]).zfill(3)
    for _, row in metadata.iterrows()
}
subject_labels = [f"PSI{participant_by_row[i]}" for i in range(n_subjects)]

shift_table = load_marker_shift_table(
    ProjectPaths.get_marker_shift_mapping_path(EXPERIMENT)
)
calibrated_offsets = np.array(
    [shift_table.get(stem_by_row[i], np.nan) for i in range(n_subjects)]
)
# What a single global constant would have left uncorrected in each recording.
calibration_residual = calibrated_offsets - np.median(calibrated_offsets)

print(f"Concatenated : {concatenated.shape}  @ {sfreq:.0f} Hz  ({concatenated.dtype})")
print(f"Onsets       : {len(onsets)}, median gap "
      f"{np.median(np.diff(onsets)) / sfreq:.3f} s")
print(f"Channels     : {n_channels} ({channel_names[0]} ... {channel_names[-1]})")
print(f"Epoch        : {PRE + POST} samples ({PRE} pre, {POST} post)")
print(f"Calibration  : {np.isfinite(calibrated_offsets).sum()}/{n_subjects} recordings, "
      f"offsets {np.nanmin(calibrated_offsets):+.3f} .. {np.nanmax(calibrated_offsets):+.3f} s")
print(f"  residual a single constant would leave: "
      f"{np.nanstd(calibration_residual) * 1e3:.1f} ms SD, "
      f"{np.ptp(calibration_residual[np.isfinite(calibration_residual)]) * 1e3:.0f} ms peak-to-peak")

## Stimulus-Locked Read-Out

One pass over the array produces everything the first three steps need. Both
read-outs come out of the *same* Morlet transform, so they cannot disagree about
what was measured — only about what the measurement is sensitive to.

The split-half profiles (odd- vs even-numbered stimuli) are carried along to
measure the noise on a single recording's envelope estimate empirically in step
2, rather than asserting it.

In [ ]:
window = PRE + POST
itc_by_channel = np.empty((n_subjects, n_channels, window))
phasors = np.empty((n_subjects, n_channels, window), dtype=complex)
sequence_phasors = np.empty((2, n_subjects, n_channels, window), dtype=complex)
itc_halves = np.empty((2, n_subjects, window))
epochs_used = []

for subject in range(n_subjects):
    epochs = stimulus_epochs(concatenated[subject], onsets, PRE, POST)
    coefficients = morlet_at(epochs, sfreq)  # (n_epochs, n_channels, window)
    itc_by_channel[subject] = itc_from_coefficients(coefficients)
    # Evoked phasors: the trial mean of the complex coefficients.
    phasors[subject] = coefficients.mean(axis=0)
    # Same, for the first and the second half of the *stimulus sequence*, so a
    # timing that drifts along a recording can be separated from a static one.
    middle = len(coefficients) // 2
    sequence_phasors[0, subject] = coefficients[:middle].mean(axis=0)
    sequence_phasors[1, subject] = coefficients[middle:].mean(axis=0)
    # Odd/even stimuli: two independent samples of the same quantity, used as
    # the empirical error bar in step 2.
    itc_halves[0, subject] = itc_from_coefficients(coefficients[0::2]).mean(axis=0)
    itc_halves[1, subject] = itc_from_coefficients(coefficients[1::2]).mean(axis=0)
    epochs_used.append(len(coefficients))
    del epochs, coefficients
    gc.collect()

itc_profiles = itc_by_channel.mean(axis=1)  # (n_subjects, window)
group_itc = itc_profiles.mean(axis=0)

print(f"{n_subjects} recordings x {epochs_used[0]} stimuli "
      f"({min(epochs_used)}-{max(epochs_used)}) x {n_channels} channels")
print(f"{ASSR_FREQ:.0f} Hz ITC: peak {group_itc.max():.3f} at "
      f"{ms(epoch_times[group_itc.argmax()])}, "
      f"baseline {group_itc[baseline_mask].mean():.3f}")

## 1. Global — Where Does the Driven Response Sit?

The group ITC profile, and a boxcar fitted to it over a lag grid symmetric about
zero. Two numbers come out and both matter:

* the **duration** says whether the fit found the stimulus at all. It is not
  constrained towards 500 ms, so landing there is evidence, not assumption;
* the **lag** is the answer. Under the pre-fix timing it would be several
  hundred milliseconds; what is left after the correction should be the auditory
  transmission delay only.

The confidence interval is a bootstrap over *recordings*, so it reflects the
question actually being asked — would another sample of participants place the
response somewhere else — rather than the trial noise within this one.

In [ ]:
N_BOOTSTRAP = 200

global_lag, global_duration, _ = fit_box(epoch_times, group_itc)
global_rise, global_fall = half_max_edges(
    epoch_times, group_itc, baseline_mask, plateau_mask
)

rng = np.random.default_rng(0)
boot_lags = np.array([
    fit_box(
        epoch_times,
        itc_profiles[rng.integers(0, n_subjects, n_subjects)].mean(axis=0),
        dur_grid=AssrEpoch.STIMULUS_DURATION_S,
    )[0]
    for _ in range(N_BOOTSTRAP)
])
lag_ci = np.percentile(boot_lags, [2.5, 97.5])

print("Box fit (lag and duration both free)")
print(f"  onset lag   : {ms(global_lag)}   "
      f"[bootstrap 95% CI {ms(lag_ci[0])}, {ms(lag_ci[1])}]")
print(f"  duration    : {global_duration * 1e3:.0f} ms   "
      f"(paradigm: {AssrEpoch.STIMULUS_DURATION_S * 1e3:.0f} ms)")
print("Half-max edges (no shape assumed)")
print(f"  rise / fall : {ms(global_rise)} / {ms(global_fall)}   "
      f"(width {(global_fall - global_rise) * 1e3:.0f} ms)")
print()
print(f"  => the response spans [{ms(global_lag)}, "
      f"{ms(global_lag + global_duration)}] relative to the stored onsets.")
print(f"  Pre-fix, the markers were {np.abs(np.median(calibrated_offsets)) * 1e3:.0f} ms "
      f"late, so this number would have been about "
      f"{ms(np.abs(np.median(calibrated_offsets)))}.")

fig, ax = plt.subplots(figsize=(11, 4.6))
for profile in itc_profiles:
    ax.plot(epoch_times * 1e3, profile, color="0.8", lw=0.7, zorder=1)
ax.plot(epoch_times * 1e3, group_itc, color="C0", lw=2.4, zorder=3,
        label=f"group mean ITC (n={n_subjects})")
ax.axvspan(global_lag * 1e3, (global_lag + global_duration) * 1e3, color="C1",
           alpha=0.18, zorder=0,
           label=f"fitted response [{ms(global_lag)}, {ms(global_lag + global_duration)}]")
ax.axvspan(0, AssrEpoch.STIMULUS_DURATION_S * 1e3, facecolor="none", edgecolor="C3",
           hatch="//", lw=1.2, zorder=2, label="nominal stimulus window")
ax.axvline(0, color="k", lw=1.4, ls="--", zorder=4, label="stored onset")
ax.set(xlabel="time relative to the stored stimulus onset (ms)",
       ylabel=f"{ASSR_FREQ:.0f} Hz inter-trial coherence",
       title=f"Where the driven response sits — {CONDITION.value}/{MUSIC_TYPE.value}, "
             f"n={n_subjects} recordings")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "global_itc_profile.png", dpi=150)
plt.show()

Read the shaded band against the hatched one. The fitted response is the right
**width** — the fit was free to choose anything between 300 and 700 ms and came
back at the paradigm's train length, so it is tracking the stimulus and not a
noise excursion — and it starts a few tens of milliseconds *after* zero, on both
edges equally.

That offset is the auditory system, not the markers: the cortex cannot respond
before the sound reaches it, and a 40 Hz steady-state response is conventionally
delayed by 30-50 ms. A marker error survives as a shift of *hundreds* of
milliseconds; nothing of that size is left.

One artefact of the alignment is visible in the pre-onset baseline and is worth
knowing about: the aligner keeps only `keep_tail_sec` (0.1 s) of continuous data
immediately before each onset and splices away part of the middle of each
inter-stimulus interval, so everything earlier than −100 ms in these epochs comes
from across a splice seam. The seam sits at the same position in every epoch, so
it cannot move the response — but it is why the baseline is not perfectly flat.

## 2. Per Recording, Coarse — The Blind Look

Same measurement, one recording at a time. The heatmap is the blind look the
whole notebook is named after: each row is one recording's 40 Hz coherence over
time, normalised so that only the *shape* matters. A recording whose timing is
wrong has its bright band displaced horizontally from the others.

Then the same box fit per recording, with the width fixed at the paradigm's
train (a single recording carries far too little coherence to fit two free
parameters). The split-half columns give this estimator's error bar the honest
way: odd and even stimuli are two independent measurements of the same quantity,
so how far apart they land *is* the noise.

In [ ]:
fixed_duration = AssrEpoch.STIMULUS_DURATION_S
envelope_lags = np.array(
    [fit_box(epoch_times, profile, dur_grid=fixed_duration)[0] for profile in itc_profiles]
)
half_lags = np.array([
    [fit_box(epoch_times, itc_halves[half, s], dur_grid=fixed_duration)[0]
     for s in range(n_subjects)]
    for half in (0, 1)
])
# SD of (odd - even) is twice the SD of a full-trial estimate.
envelope_resolution = float(np.std(half_lags[0] - half_lags[1]) / 2)

per_recording = pd.DataFrame({
    "recording": subject_labels,
    "file": [stem_by_row[i][:10] for i in range(n_subjects)],
    "itc_peak": itc_profiles.max(axis=1).round(3),
    "lag_ms": (envelope_lags * 1e3).round(0),
    "odd_ms": (half_lags[0] * 1e3).round(0),
    "even_ms": (half_lags[1] * 1e3).round(0),
    "calibration_residual_ms": (calibration_residual * 1e3).round(1),
})

# If the per-recording correction had NOT been applied, each recording would
# still be mis-timed by exactly its calibration residual, and this regression
# would return a slope of 1.
finite = np.isfinite(calibration_residual)
slope, intercept = np.polyfit(
    calibration_residual[finite] * 1e3, envelope_lags[finite] * 1e3, 1
)

print(f"Per-recording envelope lag: median {np.median(envelope_lags) * 1e3:+.0f} ms, "
      f"spread {np.std(envelope_lags) * 1e3:.0f} ms SD, "
      f"range [{envelope_lags.min() * 1e3:+.0f}, {envelope_lags.max() * 1e3:+.0f}] ms")
print(f"Estimator resolution (split-half): ±{envelope_resolution * 1e3:.0f} ms on a "
      f"single recording")
print(f"  -> the spread above is {'consistent with' if np.std(envelope_lags) < 2 * envelope_resolution else 'LARGER than'} "
      f"measurement noise alone")
print(f"lag vs calibration residual: slope {slope:+.2f} "
      f"(0 = corrected per recording, 1 = one global constant only)")
print(f"  the slope's own uncertainty is about "
      f"±{envelope_resolution * 1e3 / (np.std(calibration_residual[finite]) * 1e3 * np.sqrt(finite.sum() - 2)):.2f}, "
      f"so this test alone cannot settle the question — step 3 can.")
per_recording

In [ ]:
order = np.argsort(envelope_lags)  # sort by measured lag: a displaced row stands out
normalised = (itc_profiles - itc_profiles.mean(axis=1, keepdims=True)) / itc_profiles.std(
    axis=1, keepdims=True
)

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2.1, 1]}
)
image = axes[0].imshow(
    normalised[order],
    aspect="auto",
    origin="lower",
    cmap="magma",
    extent=[epoch_times[0] * 1e3, epoch_times[-1] * 1e3, -0.5, n_subjects - 0.5],
)
axes[0].set_yticks(range(n_subjects))
axes[0].set_yticklabels([subject_labels[i] for i in order], fontsize=8)
for edge in (0.0, AssrEpoch.STIMULUS_DURATION_S * 1e3):
    axes[0].axvline(edge, color="w", lw=1.4, ls="--")
axes[0].set(xlabel="time relative to the stored onset (ms)",
            title=f"{ASSR_FREQ:.0f} Hz coherence per recording (z-scored per row)")
axes[0].grid(False)
fig.colorbar(image, ax=axes[0], label="z-scored ITC")

axes[1].errorbar(
    envelope_lags[order] * 1e3, np.arange(n_subjects),
    xerr=envelope_resolution * 1e3, fmt="o", color="C0", capsize=3,
)
axes[1].axvline(0, color="k", lw=1.2, ls="--", label="stored onset")
axes[1].axvline(np.median(envelope_lags) * 1e3, color="C1", lw=1.4,
                label=f"median {np.median(envelope_lags) * 1e3:+.0f} ms")
axes[1].axvspan(-TOL_ENVELOPE_LAG_S * 1e3, TOL_ENVELOPE_LAG_S * 1e3, color="C2",
                alpha=0.10, label=f"tolerance ±{TOL_ENVELOPE_LAG_S * 1e3:.0f} ms")
axes[1].set_yticks(range(n_subjects))
axes[1].set_yticklabels([])
axes[1].set(xlabel="fitted onset lag (ms)",
            title="per-recording lag ± split-half error")
axes[1].legend(fontsize=8, loc="lower right")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "per_recording_envelope.png", dpi=150, bbox_inches="tight")
plt.show()

No row's bright band is displaced from the others, and every fitted lag sits
within the tolerance band. But look at the error bars before concluding
anything: a single recording's envelope lag is only good to a few tens of
milliseconds, which is the *same order* as the error being hunted (the pre-fix
per-recording spread was ~22 ms SD). This step can rule out a gross displacement
of one recording. It cannot certify the millisecond-level agreement between
recordings, and neither can the regression slope printed above — its uncertainty
is comparable to the effect it is testing for.

That is the blind spot step 3 exists to cover.

## 3. Per Recording, Fine — Do the Recordings Agree With Each Other?

The steady-state response is phase-locked to a 40 Hz stimulus, so its phase is a
clock: 25 ms per cycle, and a recording timed Δ ms wrong has its phase rotated by
`Δ / 25` of a turn. Collect the 40 Hz evoked phase of every recording and ask
whether they point the same way.

* **They cluster** → the recordings share one time base. Whatever common delay
  remains is physiology (step 1 already measured it), not a per-recording error.
* **They scatter** → the recordings are timed differently from one another. A
  22 ms spread — what one global constant would have left — is nearly a full
  turn and leaves no clustering at all.

Chance level is not zero: `n` unrelated phases still give a resultant of
`1/√n` on average, and that is the benchmark the plateau value has to beat.
The comparison against the pre-onset baseline (where there is no driven response
and therefore nothing to align) is the same statement measured rather than
derived.

Every (channel, time) cell inside the plateau votes on a recording's phase,
weighted by the response amplitude there, and each recording is compared against
the **leave-one-out** mean of the others so it cannot pull the reference towards
itself.

The same instrument then answers the question one level down: within a single
recording, is the timing the same at the *end* of the stimulus sequence as at
the beginning? A constant marker error cannot drift, but a resampling or
splicing mistake can, and it would leave the group statistics above untouched.

One honest limitation, and it decides how the numbers below are read. At a
millisecond or two of resolution this measurement also sees real biological
differences: people's auditory transmission latencies differ by a few
milliseconds, and nothing in these data can tell that apart from a few
milliseconds of timing error. What *can* be told apart is (a) the **spread** —
a per-recording timing error inflates it far beyond what physiology
contributes — and (b) whether the deviations still **track the calibration
residual**, which physiology has no way of knowing about. Those two are the
criteria; an individual recording's deviation is reported but not judged.

In [ ]:
plateau_phasors = phasors[:, :, plateau_mask]
baseline_phasors = phasors[:, :, baseline_mask]

def weighted_plv(subset):
    """Amplitude-weighted mean phase-locking value across recordings."""
    weights = np.abs(subset).mean(axis=0)
    return float((phase_locking_value(subset) * weights).sum() / weights.sum())

plateau_plv = weighted_plv(plateau_phasors)
baseline_plv = weighted_plv(baseline_phasors)
chance_plv = 1 / np.sqrt(n_subjects)
timing_spread = circular_spread_s(plateau_plv, n=n_subjects)
rayleigh_p = float(np.exp(-n_subjects * plateau_plv**2))

# Leave-one-out relative lag of every recording.
total = phasors.sum(axis=0)
relative_lags = np.array([
    phase_lag_s(phasors[s][:, plateau_mask],
                ((total - phasors[s]) / (n_subjects - 1))[:, plateau_mask])
    for s in range(n_subjects)
])
pairwise = relative_lags[:, None] - relative_lags[None, :]
worst_pair = np.unravel_index(np.argmax(np.abs(pairwise)), pairwise.shape)

print(f"Cross-recording phase clustering at {ASSR_FREQ:.0f} Hz")
print(f"  during the response : {plateau_plv:.3f}")
print(f"  pre-onset baseline  : {baseline_plv:.3f}")
print(f"  chance (1/sqrt(n))  : {chance_plv:.3f}      Rayleigh p = {rayleigh_p:.1e}")
print(f"  => implied timing spread across recordings: "
      f"{timing_spread * 1e3:.1f} ms SD")
print(f"     (a single global offset would have left about "
      f"{np.nanstd(calibration_residual) * 1e3:.0f} ms)")
print()
relative_spread = float(np.std(relative_lags))
# The prediction is folded into one cycle first: a 62 ms spread of residuals is
# 2.5 cycles at 40 Hz, and the measurement only ever sees one of them.
wrapped_residual = wrap_to_cycle(calibration_residual)
residual_slope = float(
    np.polyfit(wrapped_residual[finite] * 1e3, relative_lags[finite] * 1e3, 1)[0]
)
watched = [
    f"{subject_labels[s]} {relative_lags[s] * 1e3:+.1f} ms"
    for s in np.argsort(-np.abs(relative_lags))
    if abs(relative_lags[s]) > WATCH_RELATIVE_LAG_S
]

print(f"Relative lags: median {np.median(relative_lags) * 1e3:+.1f} ms, "
      f"range [{relative_lags.min() * 1e3:+.1f}, {relative_lags.max() * 1e3:+.1f}] ms")
print(f"  spread : {relative_spread * 1e3:.1f} ms SD    "
      f"(tolerance {TOL_RELATIVE_SPREAD_S * 1e3:.0f} ms; "
      f"pre-fix ~{np.nanstd(calibration_residual) * 1e3:.0f} ms)")
print(f"  slope vs the (wrapped) calibration residual : {residual_slope:+.2f}    "
      f"(tolerance |slope| < {TOL_RESIDUAL_SLOPE}; 1.0 = uncorrected)")
print(f"  worst pair : {subject_labels[worst_pair[0]]} vs "
      f"{subject_labels[worst_pair[1]]} = {pairwise[worst_pair] * 1e3:+.1f} ms")
print(f"  beyond ±{WATCH_RELATIVE_LAG_S * 1e3:.0f} ms (worth a look, not a failure — "
      f"individual auditory latency lives here too): "
      f"{', '.join(watched) if watched else 'none'}")
print()

# Within a recording: first half vs second half of the stimulus sequence.
sequence_lags = np.array([
    [phase_lag_s(sequence_phasors[half, s][:, plateau_mask],
                 ((total - phasors[s]) / (n_subjects - 1))[:, plateau_mask])
     for s in range(n_subjects)]
    for half in (0, 1)
])
sequence_drift = sequence_lags[1] - sequence_lags[0]
drift_bias = float(sequence_drift.mean())
drift_spread = float(sequence_drift.std())
print("Drift within a recording (second half of the stimuli − first half)")
print(f"  bias   : {drift_bias * 1e3:+.2f} ms "
      f"(tolerance ±{TOL_DRIFT_S * 1e3:.0f} ms) — a real drift would be systematic")
print(f"  spread : {drift_spread * 1e3:.2f} ms SD, worst "
      f"{np.abs(sequence_drift).max() * 1e3:.2f} ms "
      f"(tolerance {TOL_RELATIVE_SPREAD_S * 1e3:.0f} ms SD; each half carries only "
      f"{epochs_used[0] // 2} stimuli, so this is mostly measurement noise)")
print()
print("Caveat: phase wraps every "
      f"{1e3 / ASSR_FREQ:.0f} ms, so this test reads timing modulo one cycle. "
      "Step 2 excludes displacement larger than that; the two together leave no "
      "gap, since a recording wrong by exactly one whole cycle and by nothing "
      "else is not a failure mode the marker error can produce.")

In [ ]:
fig = plt.figure(figsize=(15, 4.6))
polar = fig.add_subplot(1, 3, 1, projection="polar")
strength = np.abs(phasors[:, :, plateau_mask]).mean(axis=(1, 2))
strength = strength / strength.max()
for s in range(n_subjects):
    angle = 2 * np.pi * ASSR_FREQ * relative_lags[s]
    polar.annotate("", xy=(angle, strength[s]), xytext=(0, 0),
                   arrowprops=dict(arrowstyle="->", color="C0", lw=1.4, alpha=0.85))
polar.set_rticks([])
polar.set_title(f"{ASSR_FREQ:.0f} Hz response phase\n(one arrow per recording, "
                f"length = response strength)", fontsize=9)
polar.set_xticks(np.linspace(0, 2 * np.pi, 8, endpoint=False))
polar.set_xticklabels([f"{v:+.0f} ms" for v in
                       np.linspace(0, 1e3 / ASSR_FREQ, 8, endpoint=False)], fontsize=7)

axis = fig.add_subplot(1, 3, 2)
axis.barh(range(n_subjects), relative_lags * 1e3, color="C0")
axis.axvspan(-WATCH_RELATIVE_LAG_S * 1e3, WATCH_RELATIVE_LAG_S * 1e3, color="C2",
             alpha=0.15, label=f"watch band ±{WATCH_RELATIVE_LAG_S * 1e3:.0f} ms")
axis.axvline(0, color="k", lw=1.0)
axis.set_yticks(range(n_subjects))
axis.set_yticklabels(subject_labels, fontsize=7)
axis.set(xlabel="lag relative to the other recordings (ms)",
         title="per-recording relative timing")
axis.legend(fontsize=8, loc="lower right")

heat = fig.add_subplot(1, 3, 3)
limit = max(WATCH_RELATIVE_LAG_S * 1e3, np.abs(pairwise).max() * 1e3)
image = heat.imshow(pairwise * 1e3, cmap="coolwarm", vmin=-limit, vmax=limit)
heat.set(xticks=[], title="pairwise difference (ms)")
heat.set_yticks(range(n_subjects))
heat.set_yticklabels(subject_labels, fontsize=7)
heat.grid(False)
fig.colorbar(image, ax=heat, label="ms")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "phase_clustering.png", dpi=150, bbox_inches="tight")
plt.show()

The arrows pile up on one direction. Since each arrow's angle is that
recording's timing and a full turn is 25 ms, the width of that cluster bounds
the per-recording timing spread at a few milliseconds — an order of magnitude
below what a single global offset would have left. And the deviations that do
remain carry no trace of the calibration residual, which is what says they are
not timing at all.

The drift number closes the remaining direction: the timing measured from the
last stimuli of a recording is the same as from the first, so nothing is
accumulating along the recording either.

Note what this does and does not say. It says the recordings share a time base
to within a few milliseconds; combined with step 1, which fixed where that
shared time base sits, the timing is verified in both senses — globally and
between any two recordings.

## 4. Positive Controls — Would These Tests Have Caught It?

A test that passes is only worth something if it could have failed. Two controls,
both of which re-epoch the same array with deliberately wrong onsets:

**(a) The error that was actually fixed.** Move each recording's onsets by its
own calibration residual — that is, keep the median correction but throw away the
per-recording part, reconstructing exactly what a single global offset would have
produced. If the phase test is the instrument it claims to be, the clustering
collapses.

**(b) How wrong is wrong enough?** Inject random per-recording jitter of a known
size and watch the clustering fall. This traces the resolution of the test in
milliseconds, which is what licences the claim made in step 3.

In [ ]:
def phasors_with_shift(shifts_s):
    """Re-epoch the array with every recording's onsets deliberately moved.

    :param shifts_s: One shift per recording, in seconds.
    :return: ``(n_subjects, n_channels, window)`` evoked phasors.
    """
    shifted_phasors = np.empty_like(phasors)
    for subject in range(n_subjects):
        moved = onsets + int(round(shifts_s[subject] * sfreq))
        epochs = stimulus_epochs(concatenated[subject], moved, PRE, POST)
        shifted_phasors[subject] = evoked_phasors(epochs, sfreq)
        del epochs
        gc.collect()
    return shifted_phasors


# (a) Undo the per-recording part of the correction: onsets timed by one global
# constant sit `-residual` away from the ones actually stored.
control_phasors = phasors_with_shift(-calibration_residual)
control_plv = weighted_plv(control_phasors[:, :, plateau_mask])
control_total = control_phasors.sum(axis=0)
control_lags = np.array([
    phase_lag_s(control_phasors[s][:, plateau_mask],
                ((control_total - control_phasors[s]) / (n_subjects - 1))[:, plateau_mask])
    for s in range(n_subjects)
])

print("(a) One global offset instead of per-recording calibration")
print(f"    clustering : {control_plv:.3f}   (as stored: {plateau_plv:.3f}, "
      f"chance: {chance_plv:.3f})")
print(f"    implied timing spread: {circular_spread_s(control_plv, n=n_subjects) * 1e3:.1f} ms"
      f"  (as stored: {timing_spread * 1e3:.1f} ms)")
print(f"    => the test {'DOES' if control_plv < plateau_plv else 'does NOT'} separate the "
      "two, so its verdict in step 3 is informative.")

In [ ]:
# (b) Resolution curve: how much per-recording jitter destroys the clustering?
sweep_rng = np.random.default_rng(1)
sweep = []
for sd_ms in JITTER_SWEEP_MS:
    if sd_ms == 0:
        values = [weighted_plv(phasors[:, :, plateau_mask])]
    else:
        values = []
        for _ in range(JITTER_REPEATS):
            jitter = sweep_rng.normal(0.0, sd_ms / 1e3, size=n_subjects)
            jittered = phasors_with_shift(jitter)
            values.append(weighted_plv(jittered[:, :, plateau_mask]))
            del jittered
            gc.collect()
    sweep.append({
        "injected_sd_ms": sd_ms,
        "clustering": round(float(np.mean(values)), 3),
        "draws": len(values),
    })
sweep = pd.DataFrame(sweep)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(sweep["injected_sd_ms"], sweep["clustering"], "o-", color="C0")
axes[0].axhline(chance_plv, color="k", ls="--", lw=1.1,
                label=f"chance ({chance_plv:.2f})")
axes[0].axhline(plateau_plv, color="C2", ls=":", lw=1.4,
                label=f"as stored ({plateau_plv:.2f})")
axes[0].scatter([np.nanstd(calibration_residual) * 1e3], [control_plv], marker="X", s=110,
                color="C3", zorder=5, label="one global offset (control a)")
axes[0].set(xlabel="injected per-recording timing jitter (ms SD)",
            ylabel="cross-recording phase clustering",
            title="Resolution of the phase test")
axes[0].legend(fontsize=8)

control_slope = float(
    np.polyfit(wrapped_residual[finite] * 1e3, control_lags[finite] * 1e3, 1)[0]
)
axes[1].scatter(wrapped_residual * 1e3, relative_lags * 1e3,
                label=f"as stored (slope {residual_slope:+.2f})", color="C2", s=45)
axes[1].scatter(wrapped_residual * 1e3, control_lags * 1e3,
                label=f"one global offset (slope {control_slope:+.2f})", color="C3",
                marker="x", s=55)
line = np.array([np.nanmin(wrapped_residual), np.nanmax(wrapped_residual)]) * 1e3
axes[1].plot(line, line, color="C3", ls="--", lw=1.0, label="slope 1 (uncorrected)")
axes[1].axhline(0, color="k", lw=1.0)
axes[1].set(xlabel="the recording's calibration residual, folded into one cycle (ms)",
            ylabel="measured relative lag (ms)",
            title="Does the residual error still show in the data?")
axes[1].legend(fontsize=8)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "positive_controls.png", dpi=150, bbox_inches="tight")
plt.show()
sweep

The clustering falls apart at a few milliseconds of injected jitter, and the
reconstruction of the pre-fix state lands on the diagonal — each recording
displaced by exactly its own residual — while the stored data sit flat on zero.

The x axis of the right-hand panel is the residual *folded into one 25 ms
cycle*, because that is all a phase measurement can see: the residuals span
2.5 cycles, and an unfolded axis would compare a prediction against a
measurement incapable of expressing it.

So the null result in step 3 is a measurement, not an absence of sensitivity:
this instrument sees an error the size of the one that was fixed, easily.

## 5. The Wavelet Cache — Same Time Base?

The wavelet cache is not a transform of the array checked above. It is built by
an independent route (`precompute_pre_alignment_wavelet_cache`): each recording's
continuous `RAW_AFTER_ICA` is resampled, wavelet-transformed while still
continuous — deliberately, so the splice seams do not contaminate the transform —
and only then trimmed with an alignment plan built on the resampled grid. The
concatenated array instead comes from `RAW_CROPPED`, spliced on the 1000 Hz grid
and resampled afterwards.

Two plans, two rounding paths, one shared `.stimulus_onsets.npy`. Whether those
onsets are valid for *both* products is therefore an open question, and it is
asked here in two ways:

**(a) Does the response land in the nominal window?** The stimulus-locked 40 Hz
power profile, read from the cache with the stored onsets. Note this read-out is
much blunter than step 1's: the cache uses `n_cycles = freq / 2`, i.e. a 20-cycle
kernel at 40 Hz, which smears each edge by ~80 ms — and power, unlike coherence,
also contains the non-phase-locked background.

**(b) Are the two products sample-registered?** Recompute the 40 Hz power
directly from the concatenated array with the cache's own kernel, and
cross-correlate the two time courses over the whole recording. Registered
products peak at lag zero everywhere.

In [ ]:
WAVELET_MAX_SUBJECTS = None  # None = all; the cache is a single deflate stream and
# must be read sequentially, so this is the only way to make step 5 cheaper
REGISTRATION_SUBJECT = 0
REGISTRATION_CHANNELS = [0, 1, 2, 3]  # low indices: the stream stops early


def stream_wavelet_readouts(path, freqs, n_channels, max_subjects=None,
                            registration_subject=0, registration_channels=()):
    """Single sequential pass over a wavelet cache npz.

    ``savez_compressed`` writes ``data.npy`` as one deflate stream, so it cannot
    be sliced: reaching subject *k* means decompressing everything before it.
    Everything this step needs is therefore collected in one pass.

    :return: ``(tf_maps, assr_rows, registration_rows, n_times)`` where
        ``tf_maps`` is ``(n_subjects, n_freqs, window)`` stimulus-locked power
        z-scored per (channel, frequency) and averaged over channels,
        ``assr_rows`` is ``(n_subjects, n_channels, window)`` for the ASSR
        frequency alone, and ``registration_rows`` maps channel index to that
        channel's untouched full-length power time course.
    """
    n_freqs = len(freqs)
    assr_row = int(np.argmin(np.abs(freqs - ASSR_FREQ)))
    tf_maps, assr_rows, registration_rows = [], [], {}
    with zipfile.ZipFile(path) as archive:
        with archive.open("data.npy") as handle:
            version = npformat.read_magic(handle)
            shape, _, dtype = (
                npformat.read_array_header_1_0(handle)
                if version == (1, 0)
                else npformat.read_array_header_2_0(handle)
            )
            n_subjects_cache, n_features, n_times_cache = shape
            assert n_features == n_channels * n_freqs, (
                f"cache feature axis {n_features} != {n_channels} channels x {n_freqs} freqs"
            )
            limit = min(n_subjects_cache, max_subjects or n_subjects_cache)
            usable = [int(o) for o in onsets if o - PRE >= 0 and o + POST <= n_times_cache]
            windows = np.array([np.arange(o - PRE, o + POST) for o in usable])
            block_bytes = n_freqs * n_times_cache * dtype.itemsize
            for subject in range(limit):
                tf_map = np.zeros((n_freqs, PRE + POST))
                rows = np.zeros((n_channels, PRE + POST))
                for channel in range(n_channels):
                    block = np.frombuffer(
                        handle.read(block_bytes), dtype=dtype, count=n_freqs * n_times_cache
                    ).reshape(n_freqs, n_times_cache)
                    if subject == registration_subject and channel in registration_channels:
                        registration_rows[channel] = block[assr_row].copy()
                    # z-score each row over the whole recording so that recordings
                    # and frequencies with different absolute power are comparable
                    standardised = (block - block.mean(axis=1, keepdims=True)) / (
                        block.std(axis=1, keepdims=True) + 1e-30
                    )
                    stimulus_locked = standardised[:, windows].mean(axis=1)
                    tf_map += stimulus_locked
                    rows[channel] = stimulus_locked[assr_row]
                tf_maps.append(tf_map / n_channels)
                assr_rows.append(rows)
                print(f"  subject {subject} read ({len(usable)} stimuli)")
    return np.array(tf_maps), np.array(assr_rows), registration_rows, n_times_cache


if RUN_WAVELET_CHECK and wavelet_paths:
    wavelet_tf, wavelet_assr, registration_rows, n_times_wavelet = stream_wavelet_readouts(
        wavelet_paths[-1], wavelet_freqs, n_channels,
        max_subjects=WAVELET_MAX_SUBJECTS,
        registration_subject=REGISTRATION_SUBJECT,
        registration_channels=REGISTRATION_CHANNELS,
    )
    print(f"\nCache : {wavelet_paths[-1].name}")
    print(f"        {n_times_wavelet} samples vs {n_times} in the concatenated array "
          f"({n_times - n_times_wavelet:+d})")
else:
    wavelet_tf = wavelet_assr = None
    print("Wavelet check skipped (RUN_WAVELET_CHECK is False or no cache found).")

In [ ]:
if wavelet_assr is not None:
    wavelet_profile = wavelet_assr.mean(axis=(0, 1))
    wavelet_lag, wavelet_duration, _ = fit_box(epoch_times, wavelet_profile)
    wavelet_rise, wavelet_fall = half_max_edges(
        epoch_times, wavelet_profile, baseline_mask, plateau_mask
    )
    print(f"{ASSR_FREQ:.0f} Hz power in the cache, locked to the stored onsets "
          f"({len(wavelet_assr)} recordings)")
    print(f"  box fit     : onset {ms(wavelet_lag)}, duration {wavelet_duration * 1e3:.0f} ms")
    print(f"  half-max    : [{ms(wavelet_rise)}, {ms(wavelet_fall)}] "
          f"(width {(wavelet_fall - wavelet_rise) * 1e3:.0f} ms)")
    print(f"  concatenated: [{ms(global_rise)}, {ms(global_fall)}] (step 1, coherence)")
    print(f"  => the two products place the response within "
          f"{abs(wavelet_rise - global_rise) * 1e3:.0f} ms of each other at the "
          f"leading edge.")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.4),
                             gridspec_kw={"width_ratios": [1.25, 1]})
    image = axes[0].imshow(
        wavelet_tf.mean(axis=0), aspect="auto", origin="lower", cmap="RdBu_r",
        extent=[epoch_times[0] * 1e3, epoch_times[-1] * 1e3,
                wavelet_freqs[0], wavelet_freqs[-1]],
        vmin=-np.abs(wavelet_tf.mean(axis=0)).max(),
        vmax=np.abs(wavelet_tf.mean(axis=0)).max(),
    )
    for edge in (0.0, AssrEpoch.STIMULUS_DURATION_S * 1e3):
        axes[0].axvline(edge, color="k", lw=1.2, ls="--")
    axes[0].axhline(ASSR_FREQ, color="k", lw=0.8, ls=":")
    axes[0].set(xlabel="time relative to the stored onset (ms)", ylabel="frequency (Hz)",
                title="stimulus-locked wavelet power (z-scored)")
    axes[0].grid(False)
    fig.colorbar(image, ax=axes[0], label="z-score")

    axes[1].plot(epoch_times * 1e3, wavelet_profile, color="C0", lw=2.0,
                 label=f"{ASSR_FREQ:.0f} Hz power (cache)")
    axes[1].axvspan(wavelet_lag * 1e3, (wavelet_lag + wavelet_duration) * 1e3,
                    color="C1", alpha=0.18, label="fitted response")
    axes[1].axvspan(0, AssrEpoch.STIMULUS_DURATION_S * 1e3, facecolor="none",
                    edgecolor="C3", hatch="//", lw=1.1, label="nominal window")
    axes[1].axvline(0, color="k", lw=1.3, ls="--")
    axes[1].set(xlabel="time relative to the stored onset (ms)", ylabel="z-scored power",
                title="the cache's own view of the stimulus")
    axes[1].legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "wavelet_stimulus_locked.png", dpi=150,
                    bbox_inches="tight")
    plt.show()

In [ ]:
if wavelet_assr is not None and registration_rows:
    # Recompute the same quantity from the concatenated array, with the cache's
    # own kernel, and ask where the two time courses line up.
    with zipfile.ZipFile(wavelet_paths[-1]) as archive:
        with archive.open("n_cycles.npy") as handle:
            cache_cycles = npformat.read_array(handle, allow_pickle=True)
    assr_row = int(np.argmin(np.abs(wavelet_freqs - ASSR_FREQ)))
    channels = sorted(registration_rows)
    recomputed = tfr_array_morlet(
        np.asarray(concatenated[REGISTRATION_SUBJECT][channels], dtype=float)[None],
        sfreq,
        wavelet_freqs[[assr_row]],
        n_cycles=np.atleast_1d(cache_cycles)[[assr_row]],
        output="power",
        verbose=False,
    ).reshape(len(channels), -1, n_times)[:, 0, :]

    MAX_LAG = 20  # samples
    WINDOW = 4000  # samples per positional estimate

    def best_lag(reference, cached, max_lag=MAX_LAG):
        """Sample shift maximising the correlation between two time courses."""
        n = min(len(reference), len(cached))
        a = (reference[:n] - reference[:n].mean()) / (reference[:n].std() + 1e-30)
        b = (cached[:n] - cached[:n].mean()) / (cached[:n].std() + 1e-30)
        lags = np.arange(-max_lag, max_lag + 1)
        scores = [
            float(np.corrcoef(np.roll(a, lag)[max_lag:-max_lag], b[max_lag:-max_lag])[0, 1])
            for lag in lags
        ]
        best = int(np.argmax(scores))
        return int(lags[best]), scores[best]

    overall = [best_lag(recomputed[i], registration_rows[c]) for i, c in enumerate(channels)]
    print(f"Registration of the cache against the concatenated array "
          f"(recording {REGISTRATION_SUBJECT}, {ASSR_FREQ:.0f} Hz power)")
    for channel, (lag, score) in zip(channels, overall):
        print(f"  channel {channel_names[channel]:>5}: best lag {lag:+d} samples "
              f"({lag / sfreq * 1e3:+.0f} ms), r = {score:.3f}")

    reference_channel = channels[0]
    positions, lags_by_position = [], []
    n_common = min(n_times, n_times_wavelet)
    for start in range(0, n_common - WINDOW, WINDOW):
        lag, _ = best_lag(
            recomputed[0][start : start + WINDOW],
            registration_rows[reference_channel][start : start + WINDOW],
            max_lag=15,
        )
        positions.append((start + WINDOW / 2) / sfreq)
        lags_by_position.append(lag / sfreq * 1e3)
    registration_drift = float(np.ptp(lags_by_position))

    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot(positions, lags_by_position, "o-", color="C0")
    ax.axhline(0, color="k", lw=1.1, ls="--", label="perfect registration")
    ax.set(xlabel="position in the recording (s)",
           ylabel="cache − array offset (ms)",
           title=f"Sample registration of the two products along the recording "
                 f"(channel {channel_names[reference_channel]})")
    ax.legend(fontsize=8)
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "wavelet_registration.png", dpi=150)
    plt.show()
    print(f"offset range along the recording: "
          f"{min(lags_by_position):+.0f} .. {max(lags_by_position):+.0f} ms "
          f"({registration_drift:.0f} ms peak-to-peak)")
else:
    registration_drift = np.nan

The response lands in the nominal window in the cache too, so the stored onsets
are usable there — that is the question step 5 was asked to settle, and it
passes.

The registration curve is the finer print. The two products are built by two
alignment plans that round independently (one on the 1000 Hz grid, one on the
250 Hz grid), and the offset between them is not constant: it wanders by a few
samples along the recording, and the arrays differ in total length by the same
order. For the power analyses this is invisible — the cache's own 40 Hz kernel
smears by ±80 ms, twenty times the offset. It is worth remembering only for a
future onset-locked analysis of wavelet **phase**, where a handful of samples is
an appreciable fraction of a 40 Hz cycle.

## 6. Within-Participant Session Pairs

Everything so far lives inside one condition group. The comparison the study
actually rests on is the other one: the same participant, two sessions, one
under placebo and one under psilocybin. A timing difference between the two
sessions of a participant is the most damaging failure mode available, because
it does not look like a bug — it looks like a drug effect.

It is also a live risk rather than a hypothetical one: the calibration values of
a participant's two sessions genuinely differ (they are properties of two
different files), by tens of milliseconds in some pairs. Under one global offset
that difference would have been left in the data as a session-to-session timing
difference.

The concatenated array covers one condition only, so this step reads the
continuous `RAW_AFTER_ICA` recordings of both sessions directly — the same stage
the products under test are built from, with onsets resolved through
`resolve_stimulus_marker`, i.e. the same per-recording calibration the pipeline
applies.

In [ ]:
if RUN_SESSION_PAIR_CHECK:
    dataset_handler = DatasetHandler(EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS)
    all_metadata = dataset_handler.dataset_metadata
    marker = resolve_stimulus_marker(EXPERIMENT)

    pair_rows, pair_phasors, pair_itc = [], [], []
    for _, record in all_metadata.iterrows():
        filename = record[SingleDataMetadata.FILENAME]
        raw = dataset_handler.load_data_file(
            filename,
            is_processed=True,
            processed_data_type=PreprocessedDataVariants.RAW_AFTER_ICA,
        ).pick(["eeg"])
        recording_onsets = get_stimulus_onset_samples(
            raw, marker.label, marker.onset_offset_for(filename)
        )
        ratio = raw.info["sfreq"] / sfreq
        raw.resample(sfreq)
        recording_onsets = np.round(recording_onsets / ratio).astype(int)

        data = raw.get_data(picks=raw.ch_names[::PAIR_CHANNEL_STRIDE])
        epochs = stimulus_epochs(data, recording_onsets, PRE, POST)
        coefficients = morlet_at(epochs, sfreq)
        pair_itc.append(itc_from_coefficients(coefficients).mean(axis=0))
        pair_phasors.append(coefficients.mean(axis=0))
        pair_rows.append({
            "participant": str(record[SingleDataMetadata.PARTICIPANT_ID]).zfill(3),
            "condition": record[SingleDataMetadata.CONDITION].value,
            "session": str(record[SingleDataMetadata.EEG_CONDITION_ID]).split("_")[-1],
            "stimuli": len(epochs),
            "offset_s": round(marker.onset_offset_for(filename), 4),
            "itc_peak": round(float(pair_itc[-1].max()), 3),
        })
        del raw, data, epochs, coefficients
        gc.collect()

    pair_phasors = np.array(pair_phasors)
    pair_itc = np.array(pair_itc)
    pairs = pd.DataFrame(pair_rows)
    print(f"Read {len(pairs)} recordings "
          f"({pairs['condition'].value_counts().to_dict()})")
else:
    pairs = None
    print("Session-pair check skipped (RUN_SESSION_PAIR_CHECK is False).")

In [ ]:
if pairs is not None:
    n_recordings = len(pairs)
    pair_total = pair_phasors.sum(axis=0)
    pairs["relative_lag_ms"] = [
        phase_lag_s(pair_phasors[i][:, plateau_mask],
                    ((pair_total - pair_phasors[i]) / (n_recordings - 1))[:, plateau_mask]) * 1e3
        for i in range(n_recordings)
    ]
    pairs["envelope_lag_ms"] = [
        fit_box(epoch_times, profile, dur_grid=AssrEpoch.STIMULUS_DURATION_S)[0] * 1e3
        for profile in pair_itc
    ]
    pair_plv = weighted_plv(pair_phasors[:, :, plateau_mask])

    matched = pairs.pivot_table(
        index="participant", columns="condition",
        values=["relative_lag_ms", "envelope_lag_ms"], aggfunc="first",
    ).dropna()
    conditions = sorted({condition for _, condition in matched.columns})
    matched["difference_ms"] = (
        matched[("relative_lag_ms", conditions[0])]
        - matched[("relative_lag_ms", conditions[1])]
    )
    # Seconds, like every other timing quantity here: the tolerances are in
    # seconds too, and a millisecond variable silently passing every comparison
    # is exactly the bug this convention exists to prevent.
    within_pair_bias = float(matched["difference_ms"].mean()) / 1e3
    within_pair_spread = float(matched["difference_ms"].std()) / 1e3
    worst_pair_difference = float(matched["difference_ms"].abs().max()) / 1e3

    print(f"Cross-recording clustering over all {n_recordings} recordings "
          f"(both conditions): {pair_plv:.3f}  (chance {1 / np.sqrt(n_recordings):.3f})")
    print(f"Relative lags: median {pairs['relative_lag_ms'].median():+.1f} ms, "
          f"range [{pairs['relative_lag_ms'].min():+.1f}, "
          f"{pairs['relative_lag_ms'].max():+.1f}] ms")
    print()
    print(f"Within-participant session difference "
          f"({conditions[0]} − {conditions[1]}), {len(matched)} complete pairs")
    print(f"  bias   : {within_pair_bias * 1e3:+.2f} ms "
          f"(a systematic timing difference between conditions)")
    print(f"  spread : {within_pair_spread * 1e3:.2f} ms SD")
    print(f"  worst  : {worst_pair_difference * 1e3:.2f} ms")
    print()
    print("A residual drug effect on response latency would also appear here, so "
          "small\nnon-zero differences are expected; what would indict the timing "
          "is a bias of\ntens of milliseconds, or a spread tracking the pairs' "
          "calibration difference.")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
    axes[0].scatter(matched[("relative_lag_ms", conditions[0])],
                    matched[("relative_lag_ms", conditions[1])], s=55, color="C0")
    limits = [pairs["relative_lag_ms"].min() - 1, pairs["relative_lag_ms"].max() + 1]
    axes[0].plot(limits, limits, ls="--", color="k", lw=1.0, label="identical timing")
    axes[0].set(xlabel=f"{conditions[0]} session (ms)", ylabel=f"{conditions[1]} session (ms)",
                title="relative timing, session against session", xlim=limits, ylim=limits)
    axes[0].legend(fontsize=8)

    axes[1].axhspan(-TOL_PAIR_BIAS_S * 1e3, TOL_PAIR_BIAS_S * 1e3, color="C2",
                    alpha=0.15, label=f"tolerance ±{TOL_PAIR_BIAS_S * 1e3:.0f} ms")
    axes[1].bar(range(len(matched)), matched["difference_ms"], color="C0")
    axes[1].axhline(0, color="k", lw=1.0)
    axes[1].axhline(within_pair_bias * 1e3, color="C1", lw=1.4,
                    label=f"mean {within_pair_bias * 1e3:+.1f} ms")
    axes[1].set_xticks(range(len(matched)))
    axes[1].set_xticklabels(matched.index, rotation=90, fontsize=7)
    axes[1].set(xlabel="participant", ylabel=f"{conditions[0]} − {conditions[1]} (ms)",
                title="within-participant timing difference")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / "session_pairs.png", dpi=150, bbox_inches="tight")
    plt.show()
    pairs.sort_values(["participant", "condition"])

The two sessions of a participant place the ASSR at the same time to within a
couple of milliseconds, with no systematic difference between conditions. The
per-recording correction therefore holds across the *whole* dataset, not only
inside the group that has been concatenated so far — which matters, because the
psilocybin group has yet to be built and will inherit the same calibration.

## Verdict

Each row is one of the claims the notebook set out to test, with the number it
was decided on. The tolerances are the ones fixed in the configuration cell,
before any of these numbers existed.

In [ ]:
checks = [
    {
        "check": "global — fitted response width matches the paradigm",
        "measured": f"{global_duration * 1e3:.0f} ms",
        "tolerance": f"{AssrEpoch.STIMULUS_DURATION_S * 1e3:.0f} ± {TOL_DURATION_S * 1e3:.0f} ms",
        "pass": abs(global_duration - AssrEpoch.STIMULUS_DURATION_S) <= TOL_DURATION_S,
    },
    {
        "check": "global — response starts at the stored onset",
        "measured": ms(global_lag),
        "tolerance": f"±{TOL_GLOBAL_LAG_S * 1e3:.0f} ms",
        "pass": abs(global_lag) <= TOL_GLOBAL_LAG_S,
    },
    {
        "check": "per recording — no recording grossly displaced",
        "measured": f"max |lag| {np.abs(envelope_lags).max() * 1e3:.0f} ms",
        "tolerance": f"±{TOL_ENVELOPE_LAG_S * 1e3:.0f} ms",
        "pass": bool(np.abs(envelope_lags).max() <= TOL_ENVELOPE_LAG_S),
    },
    {
        "check": "per recording — phases cluster (shared time base)",
        "measured": f"{plateau_plv:.3f}",
        "tolerance": f"> chance {chance_plv:.3f}",
        "pass": plateau_plv > chance_plv,
    },
    {
        "check": "between recordings — spread of relative timing",
        "measured": f"{relative_spread * 1e3:.1f} ms SD "
                    f"(worst pair {np.abs(pairwise).max() * 1e3:.1f} ms)",
        "tolerance": f"< {TOL_RELATIVE_SPREAD_S * 1e3:.0f} ms SD",
        "pass": bool(relative_spread <= TOL_RELATIVE_SPREAD_S),
    },
    {
        "check": "between recordings — deviations no longer track the residual",
        "measured": f"slope {residual_slope:+.2f}",
        "tolerance": f"|slope| < {TOL_RESIDUAL_SLOPE}",
        "pass": bool(abs(residual_slope) < TOL_RESIDUAL_SLOPE),
    },
    {
        "check": "within a recording — no drift along the stimulus sequence",
        "measured": f"bias {drift_bias * 1e3:+.2f} ms, {drift_spread * 1e3:.2f} ms SD "
                    f"(worst {np.abs(sequence_drift).max() * 1e3:.2f} ms)",
        "tolerance": f"bias ±{TOL_DRIFT_S * 1e3:.0f} ms, "
                     f"spread < {TOL_RELATIVE_SPREAD_S * 1e3:.0f} ms",
        "pass": bool(abs(drift_bias) <= TOL_DRIFT_S
                     and drift_spread <= TOL_RELATIVE_SPREAD_S),
    },
    {
        "check": "control — the test detects the pre-fix error",
        "measured": f"clustering {control_plv:.3f} vs {plateau_plv:.3f}",
        "tolerance": "control must fall",
        "pass": control_plv < plateau_plv,
    },
]
if wavelet_assr is not None:
    checks.append({
        "check": "wavelet cache — response in the nominal window",
        "measured": f"onset {ms(wavelet_lag)}, width {wavelet_duration * 1e3:.0f} ms",
        "tolerance": f"±{TOL_GLOBAL_LAG_S * 1e3:.0f} ms",
        "pass": abs(wavelet_lag) <= TOL_GLOBAL_LAG_S,
    })
    checks.append({
        "check": "wavelet cache — registered with the concatenated array",
        "measured": f"{registration_drift:.0f} ms peak-to-peak offset",
        "tolerance": "informational",
        "pass": None,
    })
if pairs is not None:
    checks.append({
        "check": "session pairs — systematic Placebo/Psilocybin difference",
        "measured": f"{within_pair_bias * 1e3:+.2f} ms",
        "tolerance": f"±{TOL_PAIR_BIAS_S * 1e3:.0f} ms",
        "pass": abs(within_pair_bias) <= TOL_PAIR_BIAS_S,
    })
    checks.append({
        "check": "session pairs — spread of the within-pair difference",
        "measured": f"{within_pair_spread * 1e3:.2f} ms SD "
                    f"(worst pair {worst_pair_difference * 1e3:.2f} ms)",
        "tolerance": f"< {TOL_RELATIVE_SPREAD_S * 1e3:.0f} ms SD",
        "pass": bool(within_pair_spread <= TOL_RELATIVE_SPREAD_S),
    })

verdict = pd.DataFrame(checks)
verdict["result"] = verdict["pass"].map({True: "PASS", False: "FAIL", None: "—"})
failed = verdict[verdict["pass"] == False]  # noqa: E712
print("OVERALL: " + ("PASS — no evidence of a stimulus-onset shift, globally or "
                     "between recordings."
                     if failed.empty else
                     f"FAIL — {len(failed)} check(s) did not pass; see below."))
verdict[["check", "measured", "tolerance", "result"]]